# 1. Construir CSVs con cada sujeto y todos sus mensajes unificados

In [21]:
# Importar librerías
import pandas as pd
import numpy as np
import json
import os
import csv

## Cargar CSVs originales

In [4]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Rutas CSVs y JSONs
base = "/content/drive/MyDrive/tfg/corpusMentalRiskES/processed/"
paths = {
    "ED": {
        "csv": base + "ED/gold/gold_label.csv",
        "json": base + "ED/data/"
    },
    "Depression": {
        "csv": base + "Depress/gold/gold_label.csv",
        "json": base + "Depress/data/"
    },
    "Anxiety": {
        "csv": base + "Anxiety/gold/gold_label.csv",
        "json": base + "Anxiety/data/"
    }
}

# Leer los csv
df_anx = pd.read_csv(paths["Anxiety"]["csv"], sep="\t")
df_dep = pd.read_csv(paths["Depression"]["csv"], sep="\t")
df_ed = pd.read_csv(paths["ED"]["csv"], sep="\t")

## Generar DataFrames con mensajes unificados

In [6]:
def cargar_mensajes_usuarios(ruta_json_folder):
    """
    Devuelve un diccionario {user_id: [lista_de_mensajes]}
    a partir de los .json del trastorno.
    """
    mensajes = {}
    for filename in os.listdir(ruta_json_folder):
        if filename.endswith(".json"):
            user_id = filename.replace(".json", "")
            with open(os.path.join(ruta_json_folder, filename), "r", encoding="utf-8") as f:
                # Lista de diccionarios
                data = json.load(f)
                # Lista de mensajes no vacíos: tomamos m["message"] y descartamos textos vacíos o espacios
                textos = [m["message"] for m in data if m.get("message", "").strip() != ""]
                mensajes[user_id] = textos
    return mensajes

def preparar_dataframe_trastorno(df_labels, ruta_json_folder):
    """
    Construye un DataFrame con columnas:
    user_id, texto, bs
    """
    mensajes_por_usuario = cargar_mensajes_usuarios(ruta_json_folder)

    filas = []
    for _, row in df_labels.iterrows():
        user_id = str(row["nick"])
        bs = int(row["bs"])

        if user_id not in mensajes_por_usuario:
            continue  # por si hay algún usuario sin json

        # Obtener mensajes de cada usuario y unirlos
        msgs = mensajes_por_usuario[user_id]
        texto = "\n".join(msgs).strip()

        filas.append({
            "user_id": user_id,
            "texto": texto,
            "bs": bs
        })

    df_final = pd.DataFrame(filas)
    return df_final

df_anx_final = preparar_dataframe_trastorno(df_anx, paths["Anxiety"]["json"])
df_dep_final = preparar_dataframe_trastorno(df_dep, paths["Depression"]["json"])
df_ed_final = preparar_dataframe_trastorno(df_ed, paths["ED"]["json"])

# display(df_anx_final)
# print("\n")
# display(df_dep_final)
# print("\n")
# display(df_ed_final)

# 2. División train/val/test para BERT

Para entrenar y evaluar el modelo BERT, es necesario dividir el dataset en tres subconjuntos:

* **Entrenamiento** (70%)
* **Validación** (15%)
* **Test** (15%)

El conjunto de entrenamiento se utiliza para ajustar los pesos del modelo, el de validación para seleccionar hiperparámetros y evitar sobreajuste, y el de test para evaluar el rendimiento final.

Dado que el dataset está desbalanceado (especialmente en el caso del trastorno de ansiedad), la división se realiza mediante **estratificación**, utilizando la etiqueta bs como variable de estratificación. Esto asegura que la proporción entre las clases control y suffer se mantenga de forma similar en los tres subconjuntos.

La división se efectúa en dos pasos:
1.   Separar train del 30% restante.
2.   Dividir ese 30% en validación y test manteniendo también la estratificación.

In [7]:
from sklearn.model_selection import train_test_split

def dividir_dataset(df, test_size=0.15, val_size=0.15, random_state=42):
    """
    Divide el dataset en train, val y test asegurando que la proporción
    entre clases (estratificación) se mantiene en cada subconjunto.
    """

    # 1. Train vs (Val + Test)
    df_train, df_temp = train_test_split(
        df,
        test_size = test_size + val_size,    # 0.30
        stratify = df["bs"],                 # mantiene proporción de clases
        random_state = random_state
    )

    # 2. Proporción val dentro del temporal (val + test)
    val_ratio = val_size / (test_size + val_size)

    # 3. Dividir temp → val y test
    df_val, df_test = train_test_split(
        df_temp,
        test_size = 1 - val_ratio,           # 0.5 (equivalente a 15% y 15%)
        stratify = df_temp["bs"],            # estratificación de nuevo
        random_state = random_state
    )

    return df_train, df_val, df_test

# Aplicar a cada trastorno
anx_train, anx_val, anx_test = dividir_dataset(df_anx_final)
dep_train, dep_val, dep_test = dividir_dataset(df_dep_final)
ed_train,  ed_val,  ed_test  = dividir_dataset(df_ed_final)

### Comprobar que los splits se hayan realizado correctamente manteniendo las distribuciones originales

In [9]:
def imprimir_distribucion(df, nombre):
    print(f"\nDistribución de clases en {nombre}:")
    print(df["bs"].value_counts(normalize=False))      # valores absolutos
    print(df["bs"].value_counts(normalize=True).round(3))  # proporciones

imprimir_distribucion(anx_train, "Anxiety - Train")
imprimir_distribucion(anx_val,   "Anxiety - Validation")
imprimir_distribucion(anx_test,  "Anxiety - Test")

imprimir_distribucion(dep_train, "Depression - Train")
imprimir_distribucion(dep_val,   "Depression - Validation")
imprimir_distribucion(dep_test,  "Depression - Test")

imprimir_distribucion(ed_train, "ED - Train")
imprimir_distribucion(ed_val,   "ED - Validation")
imprimir_distribucion(ed_test,  "ED - Test")


Distribución de clases en Anxiety - Train:
bs
1    310
0     40
Name: count, dtype: int64
bs
1    0.886
0    0.114
Name: proportion, dtype: float64

Distribución de clases en Anxiety - Validation:
bs
1    67
0     8
Name: count, dtype: int64
bs
1    0.893
0    0.107
Name: proportion, dtype: float64

Distribución de clases en Anxiety - Test:
bs
1    66
0     9
Name: count, dtype: int64
bs
1    0.88
0    0.12
Name: proportion, dtype: float64

Distribución de clases en Depression - Train:
bs
1    233
0    116
Name: count, dtype: int64
bs
1    0.668
0    0.332
Name: proportion, dtype: float64

Distribución de clases en Depression - Validation:
bs
1    50
0    25
Name: count, dtype: int64
bs
1    0.667
0    0.333
Name: proportion, dtype: float64

Distribución de clases en Depression - Test:
bs
1    50
0    25
Name: count, dtype: int64
bs
1    0.667
0    0.333
Name: proportion, dtype: float64

Distribución de clases en ED - Train:
bs
0    134
1    100
Name: count, dtype: int64
bs
0    0.573

### Guardar los splits en CSVs

In [24]:
output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/BERT/"
os.makedirs(output_path, exist_ok=True)

anx_train.to_csv(output_path + "anxiety_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
anx_val.to_csv(output_path + "anxiety_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
anx_test.to_csv(output_path + "anxiety_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

dep_train.to_csv(output_path + "depression_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
dep_val.to_csv(output_path + "depression_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
dep_test.to_csv(output_path + "depression_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

ed_train.to_csv(output_path + "ed_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
ed_val.to_csv(output_path + "ed_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
ed_test.to_csv(output_path + "ed_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

## Análisis adicional de número de tokens

BERT tiene un límite de 512 tokens, por lo que es necesario ver si los textos superan este límite:





In [18]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Calcular los tokens de TODOS los textos del dataset
def contar_tokens(df, tokenizer):
    longitudes = []
    for texto in df["texto"]:
        tokens = tokenizer.encode(texto, add_special_tokens=True)
        longitudes.append(len(tokens))
    return longitudes

tokens_anx = contar_tokens(df_anx_final, tokenizer)
tokens_dep = contar_tokens(df_dep_final, tokenizer)
tokens_ed  = contar_tokens(df_ed_final, tokenizer)

# Ver estadísticas generales
print("Ansiedad: media tokens =", np.mean(tokens_anx), ", máx tokens =", np.max(tokens_anx))
print("Depresión: media tokens =", np.mean(tokens_dep), ", máx tokens =", np.max(tokens_dep))
print("ED: media tokens =", np.mean(tokens_ed), ", máx tokens =", np.max(tokens_ed))

Token indices sequence length is longer than the specified maximum sequence length for this model (2567 > 512). Running this sequence through the model will result in indexing errors


Ansiedad: media tokens = 1042.98 , máx tokens = 6134
Depresión: media tokens = 820.2404809619238 , máx tokens = 6457
ED: media tokens = 799.5253731343283 , máx tokens = 12754


**Problema**: Muchos de los textos del dataset superan el límite máximo de 512 tokens que puede procesar BERT-base. Esto obliga a tomar una decisión sobre cómo manejar secuencias largas.

**Opciones disponibles**:

* **Truncación durante la tokenización**: Solo se conservan los primeros 512 tokens de cada texto. Esta solución es sencilla y eficiente, pero puede implicar pérdida de información.
* **Uso de un modelo Transformer de tipo long-range**: Estos modelos pueden procesar miles de tokens, evitando la necesidad de truncar. No obstante, su entrenamiento es más costoso en términos computacionales.

**Decisión**: En este TFG se emplearán ambos enfoques:

* BERT-base con truncación a 512 tokens, y
* un modelo long-range capaz de procesar secuencias completas.

Esto permitirá evaluar hasta qué punto la truncación afecta al rendimiento y si realmente compensa utilizar modelos long-range, que implican un mayor coste computacional.



# 3. Conjunto de evaluación para LLMs

En el caso de los LLMs no es necesario tener conjuntos train y val, ya que no hay entrenamiento. Por ello, se empleará como conjunto de evaluación el dataset completo.

In [22]:
output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
os.makedirs(output_path, exist_ok=True)

df_anx_final.to_csv(output_path + "anxiety_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
df_dep_final.to_csv(output_path + "depression_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
df_ed_final.to_csv(output_path + "ed_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)